# F1 Monte Carlo Exploration
Quick exploration of FastF1 data, feature building, and a mini Monte Carlo demo.

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import plotly.express as px

from features.feature_engineering import build_features
from simulation.monte_carlo import simulate_probability_table
from utils.session_loader import load_session

In [ ]:
# Load qualifying session (uses cache)
session_data = load_session(2023, "Bahrain", "Q")
session = session_data["session"]
laps = session_data["laps"]
print(laps.head())

In [ ]:
# Quick pace visualization (per-driver lap time distributions)
clean_laps = laps.pick_quicklaps().copy()
clean_laps["LapTimeSec"] = clean_laps["LapTime"].dt.total_seconds()
fig = px.histogram(
    clean_laps,
    x="LapTimeSec",
    color="Driver",
    nbins=25,
    barmode="overlay",
    opacity=0.35,
    labels={"LapTimeSec": "Lap Time (sec)"},
    title="Lap Time Distributions - Bahrain Q 2023",
)
fig.update_layout(template="plotly_white", height=500)
fig.show()

In [ ]:
# Build driver-level features
features_df = build_features(laps)
features_df.head()

In [ ]:
# Simple mu/sigma proxies and Monte Carlo demo
mu = -features_df["median_lap_s"].to_numpy()
sigma = np.sqrt(features_df["lap_var"].fillna(0.5).to_numpy())
drivers = features_df["Driver"].tolist()

sim_result = simulate_probability_table(mu, sigma, n_sims=3000, driver_labels=drivers, random_state=42)
sim_result.probabilities.head()

In [ ]:
# Plot probability heatmap
prob_fig = px.density_heatmap(
    sim_result.probabilities.reset_index().melt(id_vars="index", var_name="Position", value_name="Prob"),
    x="Position",
    y="index",
    z="Prob",
    color_continuous_scale="magma",
    title="Finish Position Probabilities (demo)",
)
prob_fig.update_layout(height=600, yaxis_title="Driver")
prob_fig.show()